# Desmatamento na Amazônia Legal por município, estado e região

Calcula a área desmatada (km² e %) em cada município da Amazônia Legal a partir do PRODES/INPE, separando o desmatamento **histórico** (acumulado até 2007) e **recente** (2008–2025), e agrega por estado e para a região.

Autor: Murilo Henderson · [github.com/murilohenderson/desmatamento-amazonia-legal](https://github.com/murilohenderson/desmatamento-amazonia-legal)

## 1. Configuração
Ajuste a pasta do Drive, os nomes dos arquivos e a lista de municípios de estudo.

In [ ]:
!pip install -q pyogrio
import os, glob, shutil, time
import numpy as np, shapely, pyogrio
import geopandas as gpd, pandas as pd
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/'   # pasta onde estão os dados (busca em subpastas)
B = '/content/dados/'               # cópia local para leitura mais rápida
os.makedirs(B, exist_ok=True)

ARQ_MUNICIPIOS = 'municipalities_legal_amazon.shp'
ARQ_ESTADOS    = 'states_legal_amazon.shp'

# (arquivo, camada) — use None quando o arquivo tiver uma camada só
FONTES = {
    'recente_2008_2025': [
        ('yearly_deforestation_biome_amazonia_v20261016.gpkg', None),
        ('cerrado_2008_2025.gpkg', 'cerrado_2008_2025'),
    ],
    'historico_ate_2007': [
        ('accumulated_deforestation_2007_biome_amazonia_v20261016.gpkg', None),
        ('cerrado_mascara_2000_recortado_.gpkg', None),
        ('cerrado_2001_2007_recortado.gpkg', None),
    ],
}

ESTUDO = ['Marabá', 'Parauapebas', 'Altamira', 'São Félix do Xingu', 'Redenção',
          'Itaituba', 'Santarém', 'Paragominas', 'Canaã dos Carajás', 'Cachoeira do Arari']

CRS_AREA = 'ESRI:102033'  # South America Albers Equal Area

## 2. Cópia dos dados e teste de integridade

In [ ]:
def copiar(nome):
    achados = glob.glob(DRIVE + '**/' + nome, recursive=True)
    if not achados:
        raise FileNotFoundError(f'{nome} não encontrado em {DRIVE}')
    origem = achados[0]
    if nome.endswith('.shp'):
        for ext in ['.shp', '.shx', '.dbf', '.prj', '.cpg']:
            if os.path.exists(origem[:-4] + ext):
                shutil.copy(origem[:-4] + ext, B)
    elif not os.path.exists(B + nome):
        shutil.copy(origem, B)
    print('Copiado:', origem)
    return B + nome

MUNICIPIOS = copiar(ARQ_MUNICIPIOS)
ESTADOS    = copiar(ARQ_ESTADOS)
PERIODOS = {p: [(copiar(f), l) for f, l in lst] for p, lst in FONTES.items()}

erros = []
for f, layer in [(MUNICIPIOS, None), (ESTADOS, None)] + sum(PERIODOS.values(), []):
    try:
        camadas = [c[0] for c in pyogrio.list_layers(f)]
        if layer is None and len(camadas) > 1:
            erros.append(f'{f}: várias camadas {camadas}, especifique qual'); st = 'CAMADA?'
        else:
            pyogrio.read_info(f, layer=layer); st = 'OK     '
    except Exception:
        st = 'ERRO   '; erros.append(f)
    print(st, f'{os.path.getsize(f)/1e6:7.1f} MB', os.path.basename(f), layer or '')
if erros:
    raise RuntimeError(erros)

## 3. Municípios e limite da Amazônia Legal

In [ ]:
al = gpd.read_file(ESTADOS, engine='pyogrio').to_crs(CRS_AREA)
al['geometry'] = shapely.make_valid(al.geometry.to_numpy())
al = al.dissolve()[['geometry']]

mun = gpd.read_file(MUNICIPIOS, engine='pyogrio').to_crs(CRS_AREA)
mun['geometry'] = shapely.make_valid(mun.geometry.to_numpy())
COD, NOME, UF = 'geocodigo', 'nome', 'UF'
mun[UF] = mun[COD].astype(str).str[:2].map({
    '11': 'RO', '12': 'AC', '13': 'AM', '14': 'RR', '15': 'PA',
    '16': 'AP', '17': 'TO', '21': 'MA', '51': 'MT'})
mun = gpd.overlay(mun[[COD, NOME, UF, 'geometry']], al, how='intersection', keep_geom_type=True)
mun = mun.dissolve(by=[COD, NOME, UF]).reset_index()
mun['area_mun_km2'] = mun.area / 1e6
print(len(mun), 'municípios na Amazônia Legal')

mg = mun.geometry.to_numpy()
shapely.prepare(mg)
tree = shapely.STRtree(mg)

## 4. Área desmatada por município

Polígonos inteiramente contidos em um município entram com a área total; só os que cruzam divisas passam por interseção. As camadas do PRODES não se sobrepõem (separação por bioma e por período), então não há dissolve entre elas.

In [ ]:
def validar(geoms):
    inval = ~shapely.is_valid(geoms)
    if inval.any():
        geoms[inval] = shapely.make_valid(geoms[inval])
    return geoms[~shapely.is_empty(geoms) & ~shapely.is_missing(geoms)]

def area_por_mun(fontes):
    total = np.zeros(len(mun))
    for f, layer in fontes:
        t = time.time()
        g = gpd.read_file(f, layer=layer, engine='pyogrio', use_arrow=True, columns=[]).to_crs(CRS_AREA)
        d = validar(g.explode(index_parts=False).geometry.to_numpy())
        del g
        idx_d, idx_m = tree.query(d, predicate='intersects')
        dentro = shapely.contains(mg[idx_m], d[idx_d])
        a = shapely.area(d[idx_d])
        fora = ~dentro
        a[fora] = shapely.area(shapely.intersection(d[idx_d[fora]], mg[idx_m[fora]]))
        total += np.bincount(idx_m, weights=a, minlength=len(mun))
        print(f'  {os.path.basename(f)}: {len(d):,} polígonos, '
              f'{fora.sum():,} cruzando divisas, {time.time()-t:.0f}s')
        del d, idx_d, idx_m, a
    return total / 1e6

res = mun[[COD, NOME, UF, 'area_mun_km2']].set_index(COD)
for nome, fontes in PERIODOS.items():
    print('Processando', nome)
    res[f'{nome}_km2'] = area_por_mun(fontes)
res['total_km2'] = res[[f'{n}_km2' for n in PERIODOS]].sum(axis=1)

## 5. Tabelas por município, estado e Amazônia Legal

In [ ]:
cols = [c for c in res.columns if c.endswith('_km2')]
def pct(df):
    df = df.copy()
    for c in cols:
        if c != 'area_mun_km2':
            df[c.replace('_km2', '_pct')] = 100 * df[c] / df['area_mun_km2']
    return df.round(2)

tab_mun = pct(res[res[NOME].isin(ESTUDO)]).sort_values('total_km2', ascending=False)
tab_uf  = pct(res.groupby(UF)[cols].sum())
tab_al  = pct(res[cols].sum().to_frame('Amazônia Legal').T)

falta = set(ESTUDO) - set(tab_mun[NOME])
if falta: print('Não encontrados (confira grafia/maiúsculas):', falta)

REN = {COD: 'Código IBGE', NOME: 'Município', 'area_mun_km2': 'Área total (km²)',
       'historico_ate_2007_km2': 'Desmatado até 2007 (km²)', 'historico_ate_2007_pct': 'Desmatado até 2007 (%)',
       'recente_2008_2025_km2': 'Desmatado 2008–2025 (km²)', 'recente_2008_2025_pct': 'Desmatado 2008–2025 (%)',
       'total_km2': 'Desmatado total (km²)', 'total_pct': 'Desmatado total (%)'}
ORDEM = ['Área total (km²)', 'Desmatado até 2007 (km²)', 'Desmatado até 2007 (%)',
         'Desmatado 2008–2025 (km²)', 'Desmatado 2008–2025 (%)',
         'Desmatado total (km²)', 'Desmatado total (%)']

def fmt(df, idx):
    df = df.reset_index().rename(columns={'index': idx}).rename(columns=REN)
    return df[[c for c in df.columns if c not in ORDEM] + ORDEM]

out_mun, out_uf, out_al = fmt(tab_mun, 'Código IBGE'), fmt(tab_uf, 'UF'), fmt(tab_al, 'Recorte')
display(out_mun, out_uf, out_al)

## 6. Exportação

In [ ]:
saida = '/content/desmatamento_areas.xlsx'
with pd.ExcelWriter(saida) as w:
    out_mun.to_excel(w, sheet_name='Municípios', index=False)
    out_uf.to_excel(w, sheet_name='Estados (parte na AL)', index=False)
    out_al.to_excel(w, sheet_name='Amazônia Legal', index=False)
shutil.copy(saida, DRIVE)
files.download(saida)